# 分钟贡献与日内季节性

本单元面向已有分钟贡献的调用者，分别估计 RV、BPV 的多窗口每日季节曲线。每张曲线表的行是交易日，列是北京时间的分钟末端钟点。收益去季节可以选择任一种曲线；连接分钟数据、去季节及按采样区间汇总由 c02 完成。

## 输入与计算口径

估计输入是一个品种的一条连续主力分钟贡献表，包含 trading_date、bar_at 和指定贡献列。session_calendar_df 是该连续主力对应的单品种、1m Session 日历，包含 trading_date、session_start_at、session_end_at、is_night_session；调用者先用每日主力映射选择对应合约的日历。日历须覆盖完整研究范围及所需输出日，以识别 Session 的真实首次出现和夜盘阶段；不能随模型训练窗口移动而重新定义阶段起点。日历时间戳及 bar_at 使用 Asia/Shanghai 时区，交易日采用上游 trading_date。

价格准备可独立调用 compute_minute_contributions：使用 b02 的 adjusted_log_close，按 DataFrame 普通相邻行计算，不在午休、隔夜、Session 或换月处切断。已有分钟贡献时直接调用季节估计。

$$r_i=x_i-x_{i-1},\quad RV_i=r_i^2,\quad BPV_i=\frac{\pi}{2}|r_i||r_{i-1}|.$$

每日贡献先除以当日平均有效贡献：$u_{d,m}=N_d b_{d,m}/B_d$。缺失贡献不参与日总和和有效数；日总和非正时不产生日形状。先取固定窗口，再删除其中不可用值，不跨窗口借样本。每个分钟位置独立使用剩余有效历史。

跨日聚合使用原尺度 Huber 位置 M 估计，残差尺度固定为标准化 MAD，调节参数默认 1.345。全相同样本直接返回共同值；空样本、非恒定样本的零 MAD 或不收敛均记录明确原因。零因子保留。曲线最后按目标日日历中存在且估计有效的位置归一到均值 1；因子无效的位置保留缺失。

方法接口参考：[Huber 位置估计](https://www.statsmodels.org/stable/generated/statsmodels.robust.norms.estimate_location.html)、[标准化 MAD](https://www.statsmodels.org/stable/generated/statsmodels.robust.scale.mad.html)。

## 时段变化与初始期

日盘 Session 按起止钟点识别；新增 Session 独立初始化。夜盘的整体起止时段组合变化时，整个夜盘进入新阶段，不继承上一阶段历史。只有当日无夜盘时，保留之前的夜盘阶段，不把停夜盘日视为新制度。

正常曲线只使用目标交易日之前的数据。初始回填只作用于首次出现日位于训练范围内的 Session 阶段：从首次出现日起取同一段指定长度的未来样本，所有训练初始日都使用该段样本，历史完整后恢复正常滚动。估计样本及回填日期均截断于测试开始日之前；夜盘下一阶段的数据也不进入前一阶段的估计。

日期窗口采用左闭右开区间：正常窗口 $[D-L,D)$，初始窗口 $[S,\min(S+L,T))$。整数窗口表示交易日数，DateOffset 表示相应自然日期长度；缺失日期仍占用窗口长度。

定义写在 export 单元格；同名 Python 文件只加载定义。下面的合成示例显式执行，不读真实行情、不写 demo 数据、不启动实验或云任务。


In [ ]:
from __future__ import annotations

from numbers import Integral
import numpy as np
import pandas as pd
from statsmodels.robust.norms import HuberT, estimate_location
from statsmodels.robust.scale import mad

In [ ]:
SEASONALITY_FIELD_LABELS = {
    "minute_slot": "分钟末端钟点",
    "window_name": "季节窗口名",
    "session_phase": "Session 历史阶段",
    "history_start": "实际有效历史首日",
    "history_end": "实际有效历史末日",
    "sample_count": "有效历史样本数",
    "is_initial_backfill": "是否使用固定初始窗口",
    "huber_location": "归一化前的 Huber 位置估计",
    "status": "估计状态",
    "reason": "不可用原因",
}


In [ ]:
def compute_minute_contributions(adjusted_main_minute_df: pd.DataFrame) -> pd.DataFrame:
    """从时间正序复权主力分钟表构造收益、RV 和 BPV；不重排或切断相邻行。"""
    if not adjusted_main_minute_df["bar_at"].is_monotonic_increasing:
        raise ValueError("贡献计算要求 bar_at 按真实时间正序排列")
    minute_contributions_df = adjusted_main_minute_df.copy()
    minute_contributions_df["log_return"] = minute_contributions_df["adjusted_log_close"].diff()
    minute_contributions_df["rv_contribution"] = minute_contributions_df["log_return"].pow(2)
    minute_contributions_df["bpv_contribution"] = (
        np.pi / 2 * minute_contributions_df["log_return"].abs()
        * minute_contributions_df["log_return"].shift(1).abs()
    )
    return minute_contributions_df


In [ ]:
def _prepare_seasonality_inputs(minute_contributions_df, session_calendar_df, contribution_col):
    """共用日形状及 Session 阶段；仅检查本方法必需的连接和数学条件。"""
    calendar_df = session_calendar_df.loc[:, [
        "trading_date", "session_start_at", "session_end_at", "is_night_session",
    ]].copy()
    calendar_df["trading_date"] = pd.to_datetime(calendar_df["trading_date"]).dt.normalize()
    for column in ("session_start_at", "session_end_at"):
        calendar_df[column] = pd.to_datetime(calendar_df[column]).dt.tz_convert("Asia/Shanghai")
    calendar_df = calendar_df.sort_values(["trading_date", "session_start_at"])
    if calendar_df.empty:
        raise ValueError("Session 日历不能为空")
    trading_dates = pd.DatetimeIndex(calendar_df["trading_date"].unique(), name="trading_date")

    phase_starts = {}
    day_phase_ids = {}
    minute_slot_set = set()
    session_positions = []
    slot_cache = {}
    previous_night_signature = None
    night_phase_id = None
    for trading_date, day_calendar_df in calendar_df.groupby("trading_date", sort=False):
        night_df = day_calendar_df.loc[day_calendar_df["is_night_session"]]
        night_signature = tuple(
            (row.session_start_at.time(), row.session_end_at.time())
            for row in night_df.itertuples(index=False)
        )
        if night_signature and night_signature != previous_night_signature:
            night_phase_id = len(phase_starts)
            phase_starts[night_phase_id] = trading_date
            previous_night_signature = night_signature
        for row in day_calendar_df.itertuples(index=False):
            signature = (row.session_start_at.time(), row.session_end_at.time())
            if row.is_night_session:
                phase_id = night_phase_id
            else:
                if signature not in day_phase_ids:
                    day_phase_ids[signature] = len(phase_starts)
                    phase_starts[day_phase_ids[signature]] = trading_date
                phase_id = day_phase_ids[signature]
            if signature not in slot_cache:
                slot_cache[signature] = tuple(pd.date_range(
                    row.session_start_at + pd.Timedelta(minutes=1),
                    row.session_end_at, freq="min",
                ).time)
                if not slot_cache[signature]:
                    raise ValueError("Session 必须包含可用于本方法的一分钟位置")
            slots = slot_cache[signature]
            minute_slot_set.update(slots)
            session_positions.append((trading_date, phase_id, slots))
    minute_slots = pd.Index(sorted(minute_slot_set), name="minute_slot")
    phase_codes = np.full((len(trading_dates), len(minute_slots)), -1, dtype=np.int32)
    for trading_date, phase_id, slots in session_positions:
        day_position = trading_dates.get_loc(trading_date)
        slot_positions = minute_slots.get_indexer(slots)
        if (phase_codes[day_position, slot_positions] >= 0).any():
            raise ValueError("同一交易日的 Session 分钟钟点重叠，无法形成唯一每日曲线")
        phase_codes[day_position, slot_positions] = phase_id

    contribution_df = minute_contributions_df.loc[:, ["trading_date", "bar_at", contribution_col]].copy()
    contribution_df["trading_date"] = pd.to_datetime(contribution_df["trading_date"]).dt.normalize()
    contribution_df["minute_slot"] = pd.to_datetime(contribution_df["bar_at"]).dt.tz_convert(
        "Asia/Shanghai"
    ).dt.time
    contribution_values = contribution_df[contribution_col].to_numpy(dtype=float, na_value=np.nan)
    if (contribution_values[np.isfinite(contribution_values)] < 0).any():
        raise ValueError("RV/BPV 分钟贡献必须非负")
    contribution_df[contribution_col] = np.where(np.isfinite(contribution_values), contribution_values, np.nan)
    if contribution_df.duplicated(["trading_date", "minute_slot"]).any():
        raise ValueError("输入须为一条连续序列：交易日与分钟位置必须唯一")
    day_positions = trading_dates.get_indexer(contribution_df["trading_date"])
    slot_positions = minute_slots.get_indexer(contribution_df["minute_slot"])
    if (day_positions < 0).any() or (slot_positions < 0).any():
        raise ValueError("Session 日历未覆盖提供的分钟贡献")
    if (phase_codes[day_positions, slot_positions] < 0).any():
        raise ValueError("分钟贡献与目标 Session 位置无法连接")
    contribution_matrix_df = contribution_df.pivot(
        index="trading_date", columns="minute_slot", values=contribution_col,
    ).reindex(index=trading_dates, columns=minute_slots)
    daily_totals = contribution_matrix_df.sum(axis=1, min_count=1)
    valid_counts = contribution_matrix_df.count(axis=1)
    positive_totals = daily_totals.where(np.isfinite(daily_totals) & daily_totals.gt(0))
    daily_shapes_df = contribution_matrix_df.mul(valid_counts, axis=0).div(positive_totals, axis=0)
    return daily_shapes_df, phase_codes, phase_starts


def _huber_seasonal_location(samples, *, huber_t, huber_tol, huber_maxiter):
    """有效原尺度样本的固定 MAD Huber 位置；退化及数值失败具有明确状态。"""
    if not len(samples):
        return np.nan, "no_valid_history", "窗口内没有有效历史样本"
    if np.all(samples == samples[0]):
        return float(samples[0]), "ok", ""
    scale = float(mad(samples))
    if not np.isfinite(scale) or scale <= 0:
        return np.nan, "zero_scale", "非恒定样本的标准化 MAD 为零或无效"
    try:
        with np.errstate(divide="raise", invalid="raise", over="raise"):
            location = float(estimate_location(
                samples, scale, norm=HuberT(t=huber_t),
                maxiter=huber_maxiter, tol=huber_tol,
            ))
    except ValueError as error:
        return np.nan, "not_converged", str(error)
    except (FloatingPointError, OverflowError) as error:
        return np.nan, "numerical_failure", str(error)
    if not np.isfinite(location) or location < 0:
        return np.nan, "invalid_estimate", "Huber 位置估计非有限或为负"
    return location, "ok", ""


def _normalize_seasonality_curve(raw_curves_df, diagnostics_df, expected_positions):
    """只在目标日存在且有效的位置归一化；共用于滚动和回填结果。"""
    raw_values = raw_curves_df.to_numpy(copy=True)
    raw_values[~expected_positions] = np.nan
    valid_counts = np.isfinite(raw_values).sum(axis=1)
    curve_means = np.divide(
        np.nansum(raw_values, axis=1), valid_counts,
        out=np.full(len(raw_values), np.nan), where=valid_counts > 0,
    )
    valid_means = np.isfinite(curve_means) & (curve_means > 0)
    normalized_values = np.divide(
        raw_values, curve_means[:, None], out=np.full_like(raw_values, np.nan),
        where=valid_means[:, None],
    )
    invalid_dates = raw_curves_df.index[~valid_means]
    invalid_estimates = diagnostics_df["trading_date"].isin(invalid_dates) & diagnostics_df["status"].eq("ok")
    diagnostics_df.loc[invalid_estimates, "status"] = "invalid_curve_mean"
    diagnostics_df.loc[invalid_estimates, "reason"] = "每日曲线的有效均值非正或无效"
    return pd.DataFrame(normalized_values, index=raw_curves_df.index, columns=raw_curves_df.columns), diagnostics_df


## 函数调用与结果

estimate_intraday_seasonality 一次处理一种贡献、多种窗口；output_dates 默认取全部输入日历交易日，也可只输出指定日期。当日只有日历、尚无贡献时仍可生成开盘前曲线。返回普通字典：curves 按窗口名取每日矩阵，diagnostics 是按窗口／交易日／分钟位置记录的长表，parameters 保存贡献列及数学设置。

diagnostics 中的 huber_location 是均值归一化之前的成分，供初始回填重新组合每日曲线；最终使用 curves。history_start、history_end 是实际有效样本的首末日，sample_count 是固定窗口内去除无效值后剩余样本数。失败位置在曲线中保持缺失，原因写入 status、reason；其他位置继续估计。

backfill_initial_seasonality 沿用正常结果的窗口与 Huber 参数。它覆盖训练初始期的对应 Session 位置，即使正常结果已有少量历史估计；回填后重新归一化每日曲线。原结果及输入均不修改。该函数只在同一输入、日历与参数下使用，生成身份核对由调用方负责。

大实验先生成全局正常曲线，再按每次训练边界取得回填版本；历史各日使用各自每日曲线，测试日使用正常因果曲线。正式训练按整体训练边界取得回填版本后构造既定采样序列，逐点模型重估不重复估计季节曲线。


In [ ]:
def estimate_intraday_seasonality(
    minute_contributions_df: pd.DataFrame,
    session_calendar_df: pd.DataFrame,
    *,
    contribution_col: str,
    lookbacks: dict,
    output_dates=None,
    huber_t: float = 1.345,
    huber_tol: float = 1e-6,
    huber_maxiter: int = 30,
) -> dict:
    """按交易日滚动估计一种贡献的多窗口曲线；此操作不使用未来贡献。

    lookbacks 为窗口名到正整数交易日数或正向 pd.DateOffset 的映射。
    output_dates 默认为全部日历交易日，可指定历史日或只有日历的下一日。
    返回 curves、diagnostics、parameters；曲线按窗口名选择。
    """
    if not isinstance(lookbacks, dict) or not lookbacks:
        raise ValueError("lookbacks 须为非空的窗口名到长度的映射")
    if not np.isfinite(huber_t) or huber_t <= 0 or not np.isfinite(huber_tol) or huber_tol <= 0:
        raise ValueError("Huber 阈值和容差须为有限正数")
    if isinstance(huber_maxiter, bool) or not isinstance(huber_maxiter, Integral) or huber_maxiter < 1:
        raise ValueError("huber_maxiter 须为正整数")
    daily_shapes_df, phase_codes, _ = _prepare_seasonality_inputs(
        minute_contributions_df, session_calendar_df, contribution_col,
    )
    trading_dates, minute_slots = daily_shapes_df.index, daily_shapes_df.columns
    for window_name, lookback in lookbacks.items():
        if not isinstance(window_name, str) or not window_name:
            raise ValueError("窗口名须为非空字符串")
        if isinstance(lookback, Integral) and not isinstance(lookback, bool):
            if lookback > 0:
                continue
        elif type(lookback) is pd.DateOffset:
            if all(day - lookback < day < day + lookback for day in (trading_dates[0], trading_dates[-1])):
                continue
        raise ValueError("窗口长度须为正整数交易日数或正向 pd.DateOffset")
    target_dates = trading_dates if output_dates is None else pd.DatetimeIndex(
        pd.to_datetime(output_dates), name="trading_date",
    ).normalize().unique().sort_values()
    target_positions = trading_dates.get_indexer(target_dates)
    if (target_positions < 0).any():
        raise ValueError("日历须覆盖全部输出交易日")

    shape_values = daily_shapes_df.to_numpy()
    curves_by_window = {}
    diagnostic_tables = []
    for window_name, lookback in lookbacks.items():
        raw_values = np.full((len(target_dates), len(minute_slots)), np.nan)
        records = []
        for output_position, day_position in enumerate(target_positions):
            day = trading_dates[day_position]
            history_start_position = max(0, day_position - int(lookback)) if isinstance(lookback, Integral) else int(
                trading_dates.searchsorted(day - lookback)
            )
            history_positions = np.arange(history_start_position, day_position)
            for slot_position in np.flatnonzero(phase_codes[day_position] >= 0):
                phase_id = int(phase_codes[day_position, slot_position])
                valid = (phase_codes[history_positions, slot_position] == phase_id) & np.isfinite(
                    shape_values[history_positions, slot_position]
                )
                sample_positions = history_positions[valid]
                location, status, reason = _huber_seasonal_location(
                    shape_values[sample_positions, slot_position],
                    huber_t=huber_t, huber_tol=huber_tol, huber_maxiter=huber_maxiter,
                )
                raw_values[output_position, slot_position] = location
                records.append((
                    window_name, day, minute_slots[slot_position], phase_id,
                    trading_dates[sample_positions[0]] if len(sample_positions) else pd.NaT,
                    trading_dates[sample_positions[-1]] if len(sample_positions) else pd.NaT,
                    len(sample_positions), False, location, status, reason,
                ))
        diagnostic_df = pd.DataFrame(records, columns=[
            "window_name", "trading_date", "minute_slot", "session_phase",
            "history_start", "history_end", "sample_count", "is_initial_backfill",
            "huber_location", "status", "reason",
        ])
        raw_curves_df = pd.DataFrame(raw_values, index=target_dates, columns=minute_slots)
        curves_by_window[window_name], diagnostic_df = _normalize_seasonality_curve(
            raw_curves_df, diagnostic_df, phase_codes[target_positions] >= 0,
        )
        diagnostic_tables.append(diagnostic_df)
    return {
        "curves": curves_by_window,
        "diagnostics": pd.concat(diagnostic_tables, ignore_index=True),
        "parameters": {
            "contribution_col": contribution_col, "lookbacks": dict(lookbacks),
            "huber_t": huber_t, "huber_tol": huber_tol, "huber_maxiter": int(huber_maxiter),
        },
    }


In [ ]:
def backfill_initial_seasonality(
    seasonality_result: dict,
    minute_contributions_df: pd.DataFrame,
    session_calendar_df: pd.DataFrame,
    *,
    train_start,
    test_start,
) -> dict:
    """固定初始窗口覆盖训练初始期；返回副本，测试曲线保持正常因果结果。

    传入同一次正常估计所用的贡献与日历；窗口及 Huber 设置沿用 parameters。
    训练区间为 [train_start, test_start)，不在滚动模型训练首日重置阶段。
    """
    if "training_period" in seasonality_result:
        raise ValueError("回填输入须为 estimate_intraday_seasonality 返回的正常因果结果")
    train_start, test_start = pd.Timestamp(train_start).normalize(), pd.Timestamp(test_start).normalize()
    if pd.isna(train_start) or pd.isna(test_start) or train_start >= test_start:
        raise ValueError("训练边界须满足 train_start < test_start")
    parameters = seasonality_result["parameters"]
    daily_shapes_df, phase_codes, phase_starts = _prepare_seasonality_inputs(
        minute_contributions_df, session_calendar_df, parameters["contribution_col"],
    )
    trading_dates, minute_slots = daily_shapes_df.index, daily_shapes_df.columns
    shape_values = daily_shapes_df.to_numpy()
    curves_by_window = {}
    diagnostic_tables = []
    for window_name, lookback in parameters["lookbacks"].items():
        base_curves_df = seasonality_result["curves"][window_name]
        target_positions = trading_dates.get_indexer(base_curves_df.index)
        if (target_positions < 0).any() or not base_curves_df.columns.equals(minute_slots):
            raise ValueError("回填须使用与正常估计相同的日历位置")
        diagnostic_df = seasonality_result["diagnostics"].loc[
            seasonality_result["diagnostics"]["window_name"].eq(window_name)
        ].copy()
        raw_curves_df = diagnostic_df.pivot(
            index="trading_date", columns="minute_slot", values="huber_location",
        ).reindex(index=base_curves_df.index, columns=minute_slots)
        raw_values = raw_curves_df.to_numpy(copy=True)
        changed_dates = set()
        for phase_id, phase_start in phase_starts.items():
            if not train_start <= phase_start < test_start:
                continue
            start_position = trading_dates.get_loc(phase_start)
            if isinstance(lookback, Integral):
                initial_end_position = min(len(trading_dates), start_position + int(lookback))
                initial_history_positions = np.arange(start_position, initial_end_position)
                is_initial_target = (target_positions >= start_position) & (target_positions < start_position + int(lookback))
            else:
                initial_end = phase_start + lookback
                initial_history_positions = np.arange(start_position, trading_dates.searchsorted(initial_end))
                is_initial_target = (base_curves_df.index >= phase_start) & (base_curves_df.index < initial_end)
            initial_history_positions = initial_history_positions[
                trading_dates[initial_history_positions] < test_start
            ]
            initial_targets = np.flatnonzero(
                is_initial_target & (base_curves_df.index >= train_start) & (base_curves_df.index < test_start)
                & (phase_codes[target_positions] == phase_id).any(axis=1)
            )
            if not len(initial_targets):
                continue
            initial_slot_positions = np.flatnonzero((phase_codes[target_positions[initial_targets]] == phase_id).any(axis=0))
            for slot_position in initial_slot_positions:
                valid = (phase_codes[initial_history_positions, slot_position] == phase_id) & np.isfinite(
                    shape_values[initial_history_positions, slot_position]
                )
                sample_positions = initial_history_positions[valid]
                location, status, reason = _huber_seasonal_location(
                    shape_values[sample_positions, slot_position],
                    huber_t=parameters["huber_t"], huber_tol=parameters["huber_tol"],
                    huber_maxiter=parameters["huber_maxiter"],
                )
                affected_outputs = initial_targets[
                    phase_codes[target_positions[initial_targets], slot_position] == phase_id
                ]
                affected_dates = base_curves_df.index[affected_outputs]
                raw_values[affected_outputs, slot_position] = location
                changed_dates.update(affected_dates)
                affected_diagnostics = diagnostic_df["trading_date"].isin(affected_dates) & diagnostic_df["minute_slot"].eq(
                    minute_slots[slot_position]
                )
                diagnostic_df.loc[affected_diagnostics, [
                    "history_start", "history_end", "sample_count", "is_initial_backfill",
                    "huber_location", "status", "reason",
                ]] = [
                    trading_dates[sample_positions[0]] if len(sample_positions) else pd.NaT,
                    trading_dates[sample_positions[-1]] if len(sample_positions) else pd.NaT,
                    len(sample_positions), True, location, status, reason,
                ]
            # 原来无法归一化的其他有效位置也随整日曲线重新判定。
        if changed_dates:
            restore_ok = diagnostic_df["trading_date"].isin(changed_dates) & diagnostic_df["status"].eq("invalid_curve_mean")
            diagnostic_df.loc[restore_ok, "status"] = "ok"
            diagnostic_df.loc[restore_ok, "reason"] = ""
            normalized_df, diagnostic_df = _normalize_seasonality_curve(
                pd.DataFrame(raw_values, index=base_curves_df.index, columns=minute_slots),
                diagnostic_df, phase_codes[target_positions] >= 0,
            )
            curves_df = base_curves_df.copy()
            curves_df.loc[list(changed_dates)] = normalized_df.loc[list(changed_dates)]
        else:
            curves_df = base_curves_df.copy()
        curves_by_window[window_name] = curves_df
        diagnostic_tables.append(diagnostic_df)
    return {
        "curves": curves_by_window,
        "diagnostics": pd.concat(diagnostic_tables, ignore_index=True),
        "parameters": {**parameters, "lookbacks": dict(parameters["lookbacks"])},
        "training_period": {"train_start": train_start, "test_start": test_start},
    }


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64
import hashlib
import json
import os
import platform
import sys
import pyarrow as pa
import pyarrow.parquet as pq
from R04_Research.a01_Methods.b02_MainContinuousAdjustment.c01_MainContinuousAdjustment import arrow_content_sha256


def _real_demo_identity(root, producer, upstream, parameters, scope):
    """这四个单元的实际计算定义、调用配置和上游身份；不保存模型状态。"""
    root, producer = Path(root), Path(producer)
    names = ["c01_IntradaySeasonality"]
    if producer.stem != names[0]:
        names.append("c02_ModelObservations")
    if producer.stem in ("c03_ReturnModels", "c04_VolatilityModels"):
        names.append("c03_ReturnModels")
    if producer.stem == "c04_VolatilityModels":
        names.append(producer.stem)
    source_paths = {name: producer.parent / (name + ".ipynb") for name in names}
    if producer.stem != "c01_IntradaySeasonality":
        source_paths["sampling_positions"] = producer.parent.parent / "b03_Sampling/c01_SamplingPositions.ipynb"
    sources = {}
    for name, source_path in source_paths.items():
        notebook = json.loads(source_path.read_text(encoding="utf8"))
        source_tag = "export" if name == "sampling_positions" else "demo-dependency"
        source = "\n\n".join("".join(c["source"]) for c in notebook["cells"]
            if c["cell_type"] == "code" and source_tag in c.get("metadata", {}).get("tags", []))
        sources[name] = hashlib.sha256(source.encode()).hexdigest()
    return dict(artifact_role="method_demo", producer_notebook=producer.relative_to(root).as_posix(),
        method_source_sha256=sources[producer.stem], dependency_method_source_sha256=sources,
        dependency_source_sha256={name: hashlib.sha256((root / path).read_bytes()).hexdigest()
            for name, path in {"notebook_loader": "R04_Research/a00_notebook_loader.py",
                               "data_contracts": "config/data_contracts.py"}.items()},
        upstream_demo_identity=upstream, parameters=parameters, scope=scope, timezone="Asia/Shanghai",
        runtime=dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__,
                     pyarrow=pa.__version__, platform=platform.platform()))


def _save_real_demo(frame, path, schema, identity, sort_keys):
    """明确局部 Arrow Schema；同身份同内容保留原文件；安装后复读核对。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(frame.loc[:, schema.names], schema=schema, preserve_index=False,
                               safe=True).replace_schema_metadata(schema.metadata).sort_by(sort_keys).combine_chunks()
    for field in schema:
        if not field.nullable and table[field.name].null_count:
            raise ValueError(f"demo 必需字段 {field.name} 为空")
    content = arrow_content_sha256(table, schema, sort_keys)
    expected = {**identity, "output_rows": table.num_rows, "output_sort_keys": sort_keys,
                "output_schema_sha256": hashlib.sha256(schema.serialize().to_pybytes()).hexdigest(),
                "output_content_sha256": content}
    expected = json.loads(json.dumps(expected, ensure_ascii=False, allow_nan=False))
    if path.exists():
        # 生产阶段检查旧文件自身完整性；当前版本由新计算结果的身份声明。
        old = _read_real_demo(path, require_current_sources=False)
        if {k: v for k, v in old[1].items() if k != "generated_at_utc"} == expected:
            return old[1]
    installed = {**expected, "generated_at_utc": datetime.now(timezone.utc).isoformat()}
    metadata = {**(schema.metadata or {}), b"demo_identity": json.dumps(installed, ensure_ascii=False,
        allow_nan=False, sort_keys=True).encode()}
    staged = path.with_name(path.name + ".staging")
    if staged.exists():
        raise RuntimeError(f"存在未处理 staging：{staged}")
    pq.write_table(table.replace_schema_metadata(metadata), staged, compression="zstd")
    _read_real_demo(staged, schema)
    os.replace(staged, path)
    _, actual = _read_real_demo(path, schema)
    assert actual == installed
    return installed


def _read_real_demo(path, schema=None, *, require_current_sources=True):
    path = Path(path).resolve()
    table = pq.read_table(path).combine_chunks()
    identity = json.loads(table.schema.metadata[b"demo_identity"])
    for name, expected_hash in (identity["dependency_method_source_sha256"].items() if require_current_sources else ()):
        source_path = (path.parent.parent.parent / "b03_Sampling/c01_SamplingPositions.ipynb"
            if name == "sampling_positions" else path.parent.parent / (name + ".ipynb"))
        notebook = json.loads(source_path.read_text(encoding="utf8"))
        source_tag = "export" if name == "sampling_positions" else "demo-dependency"
        source = "\n\n".join("".join(c["source"]) for c in notebook["cells"]
            if c["cell_type"] == "code" and source_tag in c.get("metadata", {}).get("tags", []))
        if hashlib.sha256(source.encode()).hexdigest() != expected_hash:
            raise RuntimeError(f"上游 {name} 计算定义已变化，请重算对应 demo")
    root = path.parents[4]
    for name, relative in {"notebook_loader": "R04_Research/a00_notebook_loader.py",
                           "data_contracts": "config/data_contracts.py"}.items():
        if require_current_sources and hashlib.sha256((root / relative).read_bytes()).hexdigest() != identity["dependency_source_sha256"][name]:
            raise RuntimeError(f"demo 依赖 {name} 已变化，请重算")
    base_schema = schema
    if base_schema is None:
        encoded_schema = (pq.ParquetFile(path).metadata.metadata or {}).get(b"ARROW:schema")
        if encoded_schema is None:
            raise RuntimeError(f"缺少原始 Arrow Schema：{path}")
        base_schema = pa.ipc.read_schema(pa.BufferReader(base64.b64decode(encoded_schema))).remove_metadata()
    if not table.schema.equals(base_schema, check_metadata=False):
        raise RuntimeError(f"局部 Schema 不匹配：{path}")
    # 各局部 demo Schema 使用空 metadata，身份单独嵌入。
    if hashlib.sha256(base_schema.serialize().to_pybytes()).hexdigest() != identity["output_schema_sha256"]:
        raise RuntimeError(f"局部 Schema 身份不匹配：{path}")
    if table.num_rows != identity["output_rows"] or arrow_content_sha256(
        table, base_schema, identity["output_sort_keys"]) != identity["output_content_sha256"]:
        raise RuntimeError(f"demo 实际内容不匹配：{path}")
    return table.to_pandas(), identity


In [ ]:
def _load_intraday_demo_inputs(root, *, start, end, test_start, calendar_path=None):
    """只读真实复权产物与正式日历；云端可传冻结的 silver 日历投影。"""
    candidate_root = Path(root)
    demo_base = candidate_root / "R04_Research/a01_Methods"
    demo_variety = "RB"
    demo_source_range = "train20100101-20241231_test20250101-20260930"
    demo_start_date, demo_end_date, demo_test_start = map(pd.Timestamp, (start, end, test_start))
    import pyarrow.dataset as ds
    from config.data_contracts import FUTURES_BAR_CALENDAR_SCHEMA, arrow_to_pandas

    demo_adjusted_path = demo_base / "b02_MainContinuousAdjustment/c01_MainContinuousAdjustment_DemoData" / (
        f"c01_MainContinuousAdjustment_{demo_variety}_{demo_source_range}_ratio1.10_log_common_demo.parquet")
    demo_adjusted_identity = json.loads(pq.read_schema(demo_adjusted_path).metadata[b"demo_identity"])
    assert demo_adjusted_identity["parameters"]["test_start_date"] == "2025-01-01"
    assert demo_adjusted_identity["underlying_code"] == demo_variety
    demo_scope = {
        "minute_read_trading_dates": {"start": demo_start_date.date().isoformat(), "end": demo_end_date.date().isoformat()},
        "training_trading_dates": {"start": demo_start_date.date().isoformat(), "end": (demo_test_start - pd.Timedelta(days=1)).date().isoformat()},
        "test_trading_dates": {"start": demo_test_start.date().isoformat(), "end": demo_end_date.date().isoformat()},
        "interval_closed": "both", "timezone": "Asia/Shanghai",
    }
    demo_adjusted_columns = ["trading_date", "bar_at", "main_contract_code", "adjusted_log_close"]
    demo_main = ds.dataset(demo_adjusted_path, format="parquet").to_table(
        filter=((ds.field("is_main") == True)
            & (ds.field("trading_date") >= demo_start_date.date())
            & (ds.field("trading_date") <= demo_end_date.date())),
        columns=demo_adjusted_columns).to_pandas()
    demo_main = demo_main.sort_values("bar_at").reset_index(drop=True)
    demo_scope["actual_available_trading_dates"] = {
        "start": pd.Timestamp(demo_main["trading_date"].min()).date().isoformat(),
        "end": pd.Timestamp(demo_main["trading_date"].max()).date().isoformat(),
    }

    # 显式选择实际主力映射；日历包含没有行情的日期和真实夜盘变化。
    demo_mapping_path = demo_base / "b01_MainContractSelection/c01_MainContractSelection_DemoData" / (
        f"c01_MainContractSelection_{demo_variety}_20100101-20260930_ratio1.10_demo.parquet")
    demo_mapping = pq.read_table(demo_mapping_path, columns=["trading_date", "main_contract_code"]).to_pandas()
    demo_mapping["trading_date"] = pd.to_datetime(demo_mapping["trading_date"])
    demo_mapping = demo_mapping.loc[demo_mapping["trading_date"].between(demo_start_date, demo_end_date)].copy()
    calendar_columns = ["contract_code", "trading_date", "session_start_at", "session_end_at", "is_night_session"]
    calendar_schema = pa.schema([FUTURES_BAR_CALENDAR_SCHEMA.field(name) for name in calendar_columns])
    calendar_partitions = pa.schema([FUTURES_BAR_CALENDAR_SCHEMA.field(name) for name in
        FUTURES_BAR_CALENDAR_SCHEMA.metadata[b"partition_columns"].decode().split(",")])
    if calendar_path is None:
        from config.settings import settings
        calendar_dataset = ds.dataset(settings.futures_lake_root / "silver" /
            FUTURES_BAR_CALENDAR_SCHEMA.metadata[b"table_name"].decode(), schema=FUTURES_BAR_CALENDAR_SCHEMA,
            format="parquet", partitioning=ds.partitioning(calendar_partitions, flavor="hive"))
        calendar_filter = ((ds.field("bar_frequency") == "1m") & (ds.field("exchange_code") == "XSGE")
            & (ds.field("underlying_code") == demo_variety) & (ds.field("year") >= demo_start_date.year)
            & (ds.field("year") <= demo_end_date.year) & (ds.field("trading_date") >= demo_start_date.date())
            & (ds.field("trading_date") <= demo_end_date.date()))
        calendar_table = calendar_dataset.to_table(filter=calendar_filter, columns=calendar_columns)
    else:
        # 云输入是正式 silver 日历的字段投影，冻结时按原 Schema 保存。
        calendar_table = pq.read_table(calendar_path, columns=calendar_columns)
        selected = pd.to_datetime(calendar_table['trading_date'].to_pandas()).between(demo_start_date, demo_end_date)
        calendar_table = calendar_table.filter(pa.array(selected.to_numpy()))
    calendar_df = arrow_to_pandas(calendar_table, calendar_schema)
    calendar_df["trading_date"] = pd.to_datetime(calendar_df["trading_date"])
    demo_calendar = calendar_df.merge(demo_mapping, on="trading_date", how="inner")
    demo_calendar = demo_calendar.loc[demo_calendar["contract_code"].eq(demo_calendar["main_contract_code"]),
        ["trading_date", "session_start_at", "session_end_at", "is_night_session"]].sort_values(
        ["trading_date", "session_start_at"]).reset_index(drop=True)
    demo_calendar_schema = pa.schema([FUTURES_BAR_CALENDAR_SCHEMA.field(name) for name in demo_calendar.columns])
    calendar_selected_table = pa.Table.from_pandas(demo_calendar, schema=demo_calendar_schema, preserve_index=False)
    demo_calendar_hash = arrow_content_sha256(calendar_selected_table, demo_calendar_schema,
        [("trading_date", "ascending"), ("session_start_at", "ascending")])
    print(f"真实上游：{demo_variety}；{len(demo_main):,} 个主力分钟；{len(demo_calendar):,} 个 Session", flush=True)
    return dict(main=demo_main, calendar=demo_calendar, scope=demo_scope,
        upstream={"adjustment": demo_adjusted_identity,
                  "main_calendar": {"content_sha256": demo_calendar_hash, "rows": len(demo_calendar)}})


def _compute_intraday_demo(demo_main, demo_calendar, *, start, test_start,
                           measures=("rv", "bpv"), lookbacks=None):
    """同一套真实贡献、因果季节与回填验收；只返回内存结果，不保存文件。"""
    demo_start_date, demo_test_start = pd.Timestamp(start), pd.Timestamp(test_start)
    demo_lookbacks = {"1y": pd.DateOffset(years=1), "3y": pd.DateOffset(years=3)} if lookbacks is None else lookbacks
    demo_window_names = tuple(demo_lookbacks)
    demo_seasonality, curve_diagnostics = {}, {}
    demo_contributions = compute_minute_contributions(demo_main)
    demo_log_prices = demo_main["adjusted_log_close"].to_numpy(dtype=float, na_value=np.nan)
    demo_returns = demo_contributions["log_return"].to_numpy(dtype=float, na_value=np.nan)
    np.testing.assert_allclose(demo_returns[1:], np.diff(demo_log_prices), equal_nan=True)
    np.testing.assert_allclose(demo_contributions["rv_contribution"], demo_returns ** 2, equal_nan=True)
    np.testing.assert_allclose(demo_contributions["bpv_contribution"].iloc[1:],
        np.pi / 2 * np.abs(demo_returns[1:] * demo_returns[:-1]), equal_nan=True)
    demo_contribution_schema = pa.schema([
        pa.field("trading_date", pa.date32(), nullable=False),
        pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai"), nullable=False),
        pa.field("main_contract_code", pa.string(), nullable=False),
        *[pa.field(name, pa.float64()) for name in
          ("adjusted_log_close", "log_return", "rv_contribution", "bpv_contribution")],
    ])
    demo_curve_schema = pa.schema([
        pa.field("trading_date", pa.date32(), nullable=False), pa.field("minute_slot", pa.time64("us"), nullable=False),
        pa.field("causal_factor", pa.float64()), pa.field("seasonal_factor", pa.float64()),
        pa.field("session_phase", pa.int64(), nullable=False), pa.field("history_start", pa.date32()),
        pa.field("history_end", pa.date32()), pa.field("sample_count", pa.int64(), nullable=False),
        pa.field("is_initial_backfill", pa.bool_(), nullable=False), pa.field("huber_location", pa.float64()),
        pa.field("status", pa.string(), nullable=False), pa.field("reason", pa.string(), nullable=False),
    ])
    for measure in measures:
        print(f"估计 {measure} 每日曲线：{tuple(demo_lookbacks)}", flush=True)
        causal = estimate_intraday_seasonality(demo_contributions, demo_calendar,
            contribution_col=f"{measure}_contribution", lookbacks=demo_lookbacks)
        filled = backfill_initial_seasonality(causal, demo_contributions, demo_calendar,
            train_start=demo_start_date, test_start=demo_test_start)
        demo_seasonality[measure] = filled
        assert filled["diagnostics"].loc[filled["diagnostics"]["is_initial_backfill"], "history_end"].dropna().lt(demo_test_start).all()
        for window_name in demo_window_names:
            curve = filled["curves"][window_name]
            pd.testing.assert_frame_equal(curve.loc[demo_test_start:], causal["curves"][window_name].loc[demo_test_start:])
            averages = curve.mean(axis=1).dropna()
            np.testing.assert_allclose(averages, 1.0, rtol=1e-10, atol=1e-10)
            diagnostics = filled["diagnostics"].loc[filled["diagnostics"]["window_name"].eq(window_name)].copy()
            diag_days = pd.DatetimeIndex(diagnostics["trading_date"])
            diag_slots = curve.columns.get_indexer(diagnostics["minute_slot"])
            diag_rows = curve.index.get_indexer(diag_days)
            diagnostics["seasonal_factor"] = curve.to_numpy()[diag_rows, diag_slots]
            diagnostics["causal_factor"] = causal["curves"][window_name].to_numpy()[diag_rows, diag_slots]
            curve_diagnostics[(measure, window_name)] = diagnostics
    return dict(contributions=demo_contributions, seasonality=demo_seasonality,
                curve_diagnostics=curve_diagnostics, contribution_schema=demo_contribution_schema,
                curve_schema=demo_curve_schema)


## Demo：分钟贡献与每日季节曲线

用途：从真实复权分钟价格构造收益、RV／BPV 贡献，调用既有季节估计与初始回填函数。输入为完整主力分钟及 Session 日历；输出为分钟贡献和 RV／BPV 各 1 年、3 年的“交易日 × 分钟位置”曲线及诊断。Huber 参数和回填边界沿用正式函数，不在 demo 重写算法。

完整范围为 RB 2010-01-01—2026-09-30，训练截至 2024-12-31，测试自 2025-01-01。先读取完整输入，再截取少量样本试错；完整输入保留供 DLC。完整阶段依次执行本单元、c02、c03／c04。云资源为每作业 10 个 ecs.c6.large 节点；完整季节曲线按独立输出日期分配十节点，节点内两个 CPU 分别估计 RV／BPV；每个日期仍使用完整历史和正式初始回填。

## 试错 demo

完整输入只读取一次；以下试错调用正式函数，仅在内存检查和展示。

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
candidate_root = Path.cwd().resolve()
while not (candidate_root / "config/settings.py").is_file():
    if candidate_root == candidate_root.parent:
        raise RuntimeError("找不到项目根目录；请在本项目内打开 Notebook")
    candidate_root = candidate_root.parent
sys.path[:0] = [str(candidate_root)]
from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import (
    _demo_products, _demo_open_run, _demo_upload, _demo_submit, _demo_wait, _demo_recover, _demo_read_saved,
    _read_real_demo, _load_intraday_demo_inputs, _compute_intraday_demo)
demo_stage = "seasonality"
real_inputs = _load_intraday_demo_inputs(candidate_root, start="2010-01-01", end="2026-09-30", test_start="2025-01-01")
full_inputs = dict(frames={"main": real_inputs["main"], "calendar": real_inputs["calendar"]},
                   scope=real_inputs["scope"], upstream=real_inputs["upstream"])

In [ ]:
trial_days = sorted(pd.to_datetime(full_inputs["frames"]["main"]["trading_date"]).unique())[:8]
if len(trial_days) < 3:
    raise ValueError("至少需要三个真实交易日进行因果及回填试错")
trial_main_df = full_inputs["frames"]["main"].loc[
    pd.to_datetime(full_inputs["frames"]["main"]["trading_date"]).isin(trial_days)].copy()
trial_calendar_df = full_inputs["frames"]["calendar"].loc[
    pd.to_datetime(full_inputs["frames"]["calendar"]["trading_date"]).isin(trial_days)].copy()
trial_start, trial_test_start = pd.Timestamp(trial_days[0]), pd.Timestamp(trial_days[-1])
trial_lookbacks = {"1y": pd.DateOffset(years=1), "3y": pd.DateOffset(years=3)}


In [ ]:
trial_computed = _compute_intraday_demo(trial_main_df, trial_calendar_df, start=trial_start,
    test_start=trial_test_start, lookbacks=trial_lookbacks)
trial_tables = {"分钟贡献": trial_computed["contributions"]}
trial_tables.update({f"{measure}_{window}": rows for (measure, window), rows in trial_computed["curve_diagnostics"].items()})


In [ ]:
demo_display_labels = {
    "sample_id": "sample_id（采样编号）", "sample_at": "sample_at（采样时刻）",
    "trading_date": "trading_date（交易日）", "sample_side": "sample_side（训练／测试侧）",
    "model": "model（模型）", "status": "status（状态）", "reason": "reason（原因）",
    "mean_forecast": "mean_forecast（下一步均值预测）",
    "variance_forecast": "variance_forecast（下一步条件方差）",
    "training_points": "training_points（训练采样点数）",
    "update_status": "update_status（观测更新状态）", "updated_mean_forecast": "updated_mean_forecast（更新后的下一步预测）",
    "parameters": "parameters（具名拟合系数）", "features": "features（HAR 特征）",
    "components": "components（MEM 成分）", "log_return": "log_return（区间对数收益）",
    "rv": "rv（区间平方收益贡献）", "bpv": "bpv（区间双幂变差贡献）",
    "log_return_status": "log_return_status（收益状态）", "rv_status": "rv_status（RV 状态）",
    "bpv_status": "bpv_status（BPV 状态）", "bar_at": "bar_at（分钟末时刻）",
    "rv_contribution": "rv_contribution（分钟平方收益贡献）",
    "bpv_contribution": "bpv_contribution（分钟双幂变差贡献）", "minute_slot": "minute_slot（分钟位置）",
    "seasonal_factor": "seasonal_factor（季节因子）", "causal_factor": "causal_factor（因果季节因子）",
    "is_initial_backfill": "is_initial_backfill（是否初始回填）"}
for configuration, trial_frame in trial_tables.items():
    print(configuration, "；试错结果仅在内存展示")
    selected_columns = [name for name in demo_display_labels if name in trial_frame.columns]
    display(trial_frame.loc[:, selected_columns].iloc[sorted(set(range(min(4, len(trial_frame)))) | set(range(max(0, len(trial_frame) - 4), len(trial_frame))))].rename(columns=demo_display_labels))
    if "status" in trial_frame:
        display(trial_frame.groupby("status", dropna=False).size().rename("行数").rename_axis("status（状态）").to_frame())


## 正式 demo（DLC）

按参数与输入身份复用本地完整成果或接续已有作业；没有才提交。云端只加载导出定义，跳过本地试错、提交、查询及展示。

In [ ]:
demo_parameters = dict(start="2010-01-01", end="2026-09-30", test_start="2025-01-01",
    contributions=["rv", "bpv"], lookback_years=[1, 3], huber_t=1.345, huber_tol=1e-6, huber_maxiter=30)
demo_resources = dict(region="cn-shenzhen", workspace_id="351454",
    oss_bucket="oss-pai-q00s96z5hnrhywm9ns-cn-shenzhen",
    image_uri="dsw-registry-vpc.cn-shenzhen.cr.aliyuncs.com/pai/python:3.13.9-cpu-ubuntu22.04-9e1c8c7e-1764329431",
    instance_type="ecs.c6.large", instance_count=10)
demo_recompute = False  # 默认复用／接续；仅明确重算且旧作业已确认终态时改为 True。


In [ ]:
if demo_stage != "seasonality":
    reference = full_inputs["frames"]["positions" if demo_stage == "observations" else "observations"]
    if len(reference) != 78007:
        raise RuntimeError("本次完整 demo 应覆盖 78,007 个既定采样位置")
demo_context = _demo_open_run(candidate_root, demo_stage, full_inputs, demo_parameters,
                              demo_resources, recompute=demo_recompute)
demo_recompute = False
print("本地完整成果有效，跳过云计算" if demo_context["cached"] else
      f'{demo_context["record"]["name"]} | {demo_context["record"]["state"]}')


In [ ]:
# 三个阶段共用的 Notebook DLC 操作；导入时只定义函数。
import base64
import importlib.metadata
import multiprocessing
import shutil
import time
import tempfile
from R04_Research.a01_Methods.b02_MainContinuousAdjustment.c01_MainContinuousAdjustment import ArrowContentDigest
import uuid
from concurrent.futures import ProcessPoolExecutor
from urllib.parse import urlparse


def _demo_json(path, value):
    """发布完整计算记录；与旧记录冲突时由调用者决定，不触发计算。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".staging")
    if temporary.exists():
        raise RuntimeError(f"存在未处理的记录 staging：{temporary}")
    with temporary.open("x", encoding="utf8") as stream:
        json.dump(value, stream, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def _demo_file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _demo_input_digest(table):
    """沿用上游摘要器，保持冻结输入的原始行顺序。"""
    digest = ArrowContentDigest(table.schema)
    digest.update(table)
    return digest.hexdigest()


def _demo_products(stage):
    """沿用本次 24 个 demo 的明确业务文件名。"""
    base = Path("R04_Research/a01_Methods/b04_ReturnVolatilityModeling")
    date_range = "RB_train20100101-20241231_test20250101-20260930"
    sampling = "N15_gk_money_all_contracts_log_interpolate_fixed"
    if stage == "seasonality":
        owner = "c01_IntradaySeasonality"
        names = [f"{owner}_{date_range}_contributions_demo.parquet",
                 *[f"{owner}_{date_range}_{m}_{w}_demo.parquet"
                   for m in ("rv", "bpv") for w in ("1y", "3y")]]
        return {name: (base / (owner + "_DemoData") / name).as_posix() for name in names}
    if stage == "observations":
        owner = "c02_ModelObservations"
        names = [f"{owner}_{date_range}_{sampling}_{case}_demo.parquet"
                 for case in ("none", "1y_returnrv", "1y_returnbpv", "3y_returnrv", "3y_returnbpv")]
        return {name: (base / (owner + "_DemoData") / name).as_posix() for name in names}
    if stage == "models":
        from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c03_ReturnModels import RETURN_MODEL_NAMES
        owners = ("c03_ReturnModels", "c04_VolatilityModels")
        names = [(owners[0], f"{owners[0]}_{date_range}_{sampling}_{m.replace(' ', '').replace('-', '')}_none_W500_demo.parquet")
                 for m in RETURN_MODEL_NAMES]
        names += [(owners[1], f"{owners[1]}_{date_range}_{sampling}_{v}_{m}_none_identity_W500_demo.parquet")
                  for v in ("rv", "bpv") for m in ("ARMA", "ARFIMA", "HAR", "MEM")]
        return {name: (base / (owner + "_DemoData") / name).as_posix() for owner, name in names}
    raise ValueError("未知 demo 阶段")


def _demo_source_files(root):
    root = Path(root)
    methods = ("b01_MainContractSelection/c01_MainContractSelection",
               "b02_MainContinuousAdjustment/c01_MainContinuousAdjustment",
               "b03_Sampling/c01_SamplingPositions", "b03_Sampling/c02_SamplingExecution",
               "b04_ReturnVolatilityModeling/c01_IntradaySeasonality",
               "b04_ReturnVolatilityModeling/c02_ModelObservations",
               "b04_ReturnVolatilityModeling/c03_ReturnModels",
               "b04_ReturnVolatilityModeling/c04_VolatilityModels")
    files = ["config/__init__.py", "config/data_contracts.py", "R04_Research/a00_notebook_loader.py"]
    files += ["R04_Research/a01_Methods/" + name + suffix
              for name in methods for suffix in (".ipynb", ".py")]
    contents = {}
    for relative in files:
        path = root / relative
        if path.suffix == ".ipynb":
            notebook = json.loads(path.read_text(encoding="utf8"))
            # 保留单元格位置，只有定义随代码包上传；试错数据、执行及绘图不上传。
            for cell in notebook["cells"]:
                if (cell["cell_type"] != "code" or "export" not in cell.get("metadata", {}).get("tags", [])
                        or "display-only" in cell.get("metadata", {}).get("tags", [])):
                    cell["source"] = []
                if cell["cell_type"] == "code":
                    cell["outputs"], cell["execution_count"] = [], None
                cell.pop("attachments", None)
                cell.pop("id", None)
                cell["metadata"] = {"tags": [tag for tag in cell.get("metadata", {}).get("tags", [])
                                            if tag in ("export", "demo-dependency")]}
            notebook["metadata"] = {k: v for k, v in notebook.get("metadata", {}).items()
                                    if k in ("kernelspec", "language_info")}
            contents[relative] = (json.dumps(notebook, ensure_ascii=False, indent=1) + "\n").encode()
        else:
            contents[relative] = path.read_bytes()
    return contents


def _demo_open_run(root, stage, full_inputs, parameters, resources, *, recompute=False):
    """核对本次输入／定义／成果；返回复用或接续状态，不提交。"""
    if resources["instance_count"] != 10 or resources["instance_type"] != "ecs.c6.large":
        raise ValueError("本次 demo 固定使用 10 个 ecs.c6.large 节点")
    if (parameters.get("start"), parameters.get("end"), parameters.get("test_start")) != ("2010-01-01", "2026-09-30", "2025-01-01"):
        raise ValueError("参数与本次固定输出范围不一致")
    if stage == "seasonality" and (parameters.get("contributions"), parameters.get("lookback_years"),
            parameters.get("huber_t"), parameters.get("huber_tol"), parameters.get("huber_maxiter")) != (["rv", "bpv"], [1, 3], 1.345, 1e-6, 30):
        raise ValueError("参数与本次季节曲线组合调用不一致")
    if stage == "observations" and (parameters.get("cases") != ["none", "1y_returnrv", "1y_returnbpv", "3y_returnrv", "3y_returnbpv"] or parameters.get("sampling") != dict(N=15, axis="gk_money", quantity_scope="all_contracts", sampling_method="log_interpolate", execution_mode="fixed", K=None)):
        raise ValueError("参数与既定采样和五组观测不一致")
    if stage == "models" and (parameters.get("history_points"), parameters.get("mean_order"), parameters.get("volatility_order"),
            parameters.get("fractional_memory"), parameters.get("har_windows"), parameters.get("mem_order"),
            parameters.get("transform"), parameters.get("seasonality"), parameters.get("require_nonnegative_prediction"), parameters.get("refit")) != (500, [1, 1], [1, 1], 30, [1, 5, 20], [1, 1], "identity", None, True, "every_point"):
        raise ValueError("参数与本次固定模型组合调用不一致")
    root = Path(root)
    source_files = _demo_source_files(root)
    source_hashes = {name: hashlib.sha256(content).hexdigest() for name, content in source_files.items()}
    tables = {name: pa.Table.from_pandas(frame, preserve_index=False).replace_schema_metadata(None).combine_chunks()
              for name, frame in full_inputs["frames"].items()}
    frame_identity = {}
    for name, table in tables.items():
        frame_identity[name] = dict(rows=table.num_rows, schema_sha256=hashlib.sha256(table.schema.serialize().to_pybytes()).hexdigest(),
                                   content_sha256=_demo_input_digest(table))
    requirements = [name + "==" + importlib.metadata.version(name)
                    for name in ("numpy", "pandas", "pyarrow", "polars", "scipy", "numba", "arch", "statsmodels", "matplotlib", "threadpoolctl")]
    identity = dict(stage=stage, scope=full_inputs["scope"], upstream=full_inputs["upstream"],
                    frames=frame_identity, parameters=parameters, source_files=source_hashes,
                    resources=resources, requirements=requirements)
    identity = json.loads(json.dumps(identity, sort_keys=True, allow_nan=False))
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    owner = {"seasonality": "c01_IntradaySeasonality", "observations": "c02_ModelObservations",
             "models": "c03_ReturnModels"}[stage]
    family = root / "R04_Research/a01_Methods/b04_ReturnVolatilityModeling" / (owner + "_CloudRuns") / key
    pointer = family / "current.json"
    for previous_path in family.parent.glob("*/*/job.json"):
        previous = json.loads(previous_path.read_text(encoding="utf8"))
        if (previous.get("identity_key") != key and
                previous.get("state") in ("submitting", "submission_uncertain", "submitted", "running")):
            raise RuntimeError(f"另有未确认终态的作业：{previous['name']}；先接续其记录，不因来源变化重提")
    family.mkdir(parents=True, exist_ok=True)
    creation_lock = family / "creation.lock"
    # 两本模型 Notebook 同时进入时，先锁定再读取指向，避免创建两个 UUID。
    with creation_lock.open("x", encoding="utf8") as stream:
        stream.write(key)
    try:
        record_path, record = None, None
        if pointer.exists():
            current = json.loads(pointer.read_text(encoding="utf8"))
            request_uuid = str(uuid.UUID(current["uuid"]))
            record_path = family / request_uuid / "job.json"
            if not record_path.exists():
                raise RuntimeError("已有作业指向的记录丢失；先核查，不创建替代作业")
            record = json.loads(record_path.read_text(encoding="utf8"))
            if record["identity"] != identity or record["uuid"] != request_uuid:
                raise RuntimeError("已有作业记录身份不匹配")
        elif any(family.glob("*/job.json")):
            raise RuntimeError("已有作业记录但 current 指向丢失；先核查，不重复提交")
        if recompute and record:
            if record.get("job_id") and record["state"] not in ("cloud_failed", "completed", "recovery_failed"):
                raise RuntimeError("已有云作业未确认终态，不能新建重算")
            record = None
        if record is None:
            request_uuid = str(uuid.uuid4())
            batch = datetime.now(timezone.utc).astimezone().strftime("%Y%m%d")
            key_parameters = {"seasonality": "RVBPV-W1y3y",
                              "observations": "gkMoney-N15-RVBPV-W1y3y",
                              "models": "gkMoney-N15-H500-Return6-RVBPV4"}[stage]
            name = f"Demo-RVM-{stage}-RB-20100101-20260930-{key_parameters}-{batch}-{request_uuid}"
            record_path = family / request_uuid / "job.json"
            record = dict(state="prepared", uuid=request_uuid, batch=batch, name=name, identity=identity,
                          identity_key=key, products=_demo_products(stage), created_at_utc=datetime.now(timezone.utc).isoformat())
            _demo_json(record_path, record)
            _demo_json(pointer, {"uuid": request_uuid})
    finally:
        creation_lock.unlink()
    completion_path = record_path.parent / "completion.json"
    cached = False
    if completion_path.exists():
        completion = json.loads(completion_path.read_text(encoding="utf8"))
        if completion["state"] != "completed" or completion["identity"] != identity or completion["uuid"] != record["uuid"]:
            raise RuntimeError("本地完成记录与本次完整计算不匹配")
        if set(completion["products"]) != set(record["products"]):
            raise RuntimeError("本地完成记录产物集合不匹配")
        cached = all((root / record["products"][name]).is_file()
                     and _demo_file_digest(root / record["products"][name]) == item["file_sha256"]
                     for name, item in completion["products"].items())
    return dict(root=root, stage=stage, full_inputs=full_inputs, tables=tables, source_files=source_files,
                parameters=parameters, resources=resources, requirements=requirements, identity=identity,
                record_path=record_path, record=record, cached=cached)


def _demo_pai_session(resources):
    from pai.session import setup_default_session
    from alibabacloud_tea_util.models import RuntimeOptions
    return setup_default_session(region_id=resources["region"], workspace_id=resources["workspace_id"],
        oss_bucket_name=resources["oss_bucket"], network="PUBLIC",
        oss_endpoint=f"https://oss-{resources['region']}.aliyuncs.com",
        runtime=RuntimeOptions(autoretry=False, max_attempts=1, connect_timeout=5000, read_timeout=10000))


def _demo_upload(ctx):
    """从保留的全区间内存输入冻结并上传；不生成执行脚本。"""
    record = json.loads(ctx["record_path"].read_text(encoding="utf8"))
    if ctx["cached"] or record.get("job_id") or record["state"] != "prepared":
        return
    run = ctx["record_path"].parent
    code_dir, input_dir = run / "code", run / "input"
    manifest_path = input_dir / "manifest.json"
    if not manifest_path.exists():
        if code_dir.exists() or input_dir.exists():
            raise RuntimeError("存在未完成的冻结目录；先核查，不覆盖或重新提交")
        input_dir.mkdir()
        for name, table in ctx["tables"].items():
            pq.write_table(table, input_dir / (name + ".parquet"), compression="zstd")
        manifest = dict(identity=ctx["identity"], files={p.name: _demo_file_digest(p) for p in input_dir.glob("*.parquet")})
        _demo_json(manifest_path, manifest)
        for relative, content in ctx["source_files"].items():
            destination = code_dir / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(content)
        plan = dict(stage=ctx["stage"], uuid=record["uuid"], identity_key=record["identity_key"],
                    identity=ctx["identity"], metadata={k: v for k, v in ctx["full_inputs"].items() if k != "frames"},
                    products=record["products"], input_manifest_sha256=_demo_file_digest(manifest_path))
        _demo_json(code_dir / "demo_plan.json", plan)
    manifest = json.loads(manifest_path.read_text(encoding="utf8"))
    if manifest["identity"] != ctx["identity"]:
        raise RuntimeError("冻结输入与当前全区间输入不匹配")
    for name, expected in manifest["files"].items():
        if _demo_file_digest(input_dir / name) != expected:
            raise RuntimeError("冻结输入文件变化")
    for relative, expected in ctx["identity"]["source_files"].items():
        if _demo_file_digest(code_dir / relative) != expected:
            raise RuntimeError("冻结代码变化")
    session = _demo_pai_session(ctx["resources"])
    bucket = session.oss_bucket
    prefix = f"research/method_demo/{record['identity_key']}/inputs/{record['uuid']}/"
    for path in [*sorted(input_dir.glob("*.parquet")), manifest_path]:
        key = prefix + path.name
        if not bucket.object_exists(key):
            bucket.put_object_from_file(key, str(path))
        stream, digest = bucket.get_object(key), hashlib.sha256()
        try:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                digest.update(block)
        finally:
            stream.close()
        if digest.hexdigest() != _demo_file_digest(path):
            raise RuntimeError("OSS 冻结输入内容不匹配")
    # 云节点通过只读签名 GET 下载冻结文件，不挂载大型研究输入。
    plan_path = code_dir / "demo_plan.json"
    plan = json.loads(plan_path.read_text(encoding="utf8"))
    plan["input_transport"] = "signed_https_to_local"
    plan["input_downloads"] = {
        path.name: dict(url=bucket.sign_url("GET", prefix + path.name, 86400),
                        sha256=_demo_file_digest(path))
        for path in [*sorted(input_dir.glob("*.parquet")), manifest_path]
    }
    if any(urlparse(item["url"]).scheme != "https" for item in plan["input_downloads"].values()):
        raise RuntimeError("冻结输入下载必须使用 HTTPS")
    _demo_json(plan_path, plan)
    record.update(input_uri=f"oss://{bucket.bucket_name}/{prefix}", input_transport="signed_https_to_local",
                  requested_output_uri=f"oss://{bucket.bucket_name}/research/method_demo/{record['identity_key']}/runs/{record['uuid']}/")
    _demo_json(ctx["record_path"], record)
    ctx["record"], ctx["session"] = record, session


def _demo_submit(ctx):
    from pai.estimator import Estimator
    if ctx["cached"]:
        return ctx["record"]
    record = json.loads(ctx["record_path"].read_text(encoding="utf8"))
    session = ctx.get("session") or _demo_pai_session(ctx["resources"])
    if record.get("job_id"):
        ctx["record"], ctx["session"] = record, session
        return record
    if record["state"] != "prepared":
        # 创建响应不确定时，仅按已保存的唯一名称核对，不再次 fit。
        matches = [item for item in session.training_job_api.list(training_job_name=record["name"], page_size=100).items
                   if item.get("TrainingJobName") == record["name"]]
        if len(matches) != 1:
            raise RuntimeError(f"创建结果待核查：{len(matches)} 个同名作业；保持原 UUID，不重提")
        record.update(job_id=matches[0]["TrainingJobId"], state="submitted")
        from pai.job import TrainingJob
        record["console_uri"] = TrainingJob.get(record["job_id"], session=session).console_uri
        _demo_json(ctx["record_path"], record)
        ctx["record"], ctx["session"] = record, session
        return record
    if not record.get("input_uri"):
        raise RuntimeError("完整输入尚未上传")
    lock = ctx["record_path"].parent / "submission.lock"
    with lock.open("x", encoding="utf8") as stream:
        stream.write(record["uuid"])
    module, function = {"seasonality": ("c01_IntradaySeasonality", "run_seasonality_demo_node"),
                        "observations": ("c02_ModelObservations", "run_observations_demo_node"),
                        "models": ("c03_ReturnModels", "run_models_demo_node")}[ctx["stage"]]
    import shlex
    bootstrap = ("from pathlib import Path; import os,sys; r=Path.cwd(); "
                 "sys.path[:0]=[str(r)]; "
                 f"from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.{module} import {function}; "
                 f"{function}(r,None,os.environ['PAI_OUTPUT_MODEL'],"
                 "rank=int(os.environ['RANK']),node_count=int(os.environ['WORLD_SIZE']))")
    command = "python -c " + shlex.quote(bootstrap)
    try:
        current = json.loads(ctx["record_path"].read_text(encoding="utf8"))
        if current["state"] != "prepared" or current.get("job_id"):
            raise RuntimeError("提交记录已由另一内核更新；重跑本单元接续，不重复提交")
        record = current
        record.update(state="submitting", command=command)
        _demo_json(ctx["record_path"], record)
        resources = ctx["resources"]
        estimator = Estimator(image_uri=resources["image_uri"], command=command,
            source_dir=str(ctx["record_path"].parent / "code"), job_type="PyTorchJob",
            instance_type=resources["instance_type"], instance_count=resources["instance_count"],
            environments={key: "1" for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS")},
            requirements=ctx["requirements"], max_run_time=None, base_job_name=record["name"],
            output_path=record["requested_output_uri"], session=session)
        job = estimator.fit(inputs={}, wait=False, show_logs=False, job_name=record["name"])
        record.update(state="submitted", job_id=job.id, console_uri=job.console_uri)
        _demo_json(ctx["record_path"], record)
    except BaseException as error:
        record = json.loads(ctx["record_path"].read_text(encoding="utf8"))
        if record["state"] == "submitting":
            record.update(state="submission_uncertain", error=f"{type(error).__name__}: {error}")
            _demo_json(ctx["record_path"], record)
        raise
    finally:
        lock.unlink()
    ctx["record"], ctx["session"] = record, session
    return record


def _demo_wait(ctx):
    """每 60 秒普通 Python 查询；暂时未知不改变作业身份。"""
    from pai.job import TrainingJob, TrainingJobStatus
    if ctx["cached"]:
        return
    record = json.loads(ctx["record_path"].read_text(encoding="utf8"))
    if not record.get("job_id"):
        raise RuntimeError("没有已确认的作业 ID")
    session = ctx.get("session") or _demo_pai_session(ctx["resources"])
    ctx["session"] = session
    while True:
        try:
            job = TrainingJob.get(record["job_id"], session=session)
            status = job.status
        except Exception as error:
            record.update(last_query_error=f"{type(error).__name__}: {error}",
                          last_query_at_utc=datetime.now(timezone.utc).isoformat())
            _demo_json(ctx["record_path"], record)
            print(f"{record['name']} | 状态暂未知 | {record.get('console_uri', '')}", flush=True)
            time.sleep(60)
            continue
        record.update(cloud_status=str(status), last_query_at_utc=datetime.now(timezone.utc).isoformat(),
                      console_uri=job.console_uri)
        print(f"{record['name']} | {status} | {job.console_uri}", flush=True)
        if status == TrainingJobStatus.Succeed:
            record.update(state="cloud_completed_pending_recovery", actual_output_uri=job.output_path())
        elif status in {*TrainingJobStatus.completed_status(), TrainingJobStatus.CreateFailed}:
            record.update(state="cloud_failed", error=job.reason_message or str(status))
        else:
            record["state"] = "running"
        _demo_json(ctx["record_path"], record)
        ctx["record"] = record
        if record["state"] == "cloud_failed":
            raise RuntimeError(f"云作业终止，未重跑：{record['job_id']}；{record['error']}")
        if record["state"] == "cloud_completed_pending_recovery":
            return
        time.sleep(60)


def _demo_stage_node_inputs(code_root, input_dir, rank):
    """只读签名 HTTPS 流式下载到节点本地盘；不读取挂载输入，不重试失败下载。"""
    from urllib.request import urlopen
    plan = json.loads((Path(code_root) / "demo_plan.json").read_text(encoding="utf8"))
    if plan.get("input_transport") != "signed_https_to_local":
        raise RuntimeError("本作业未冻结签名 HTTPS 下载方式")
    local = Path(tempfile.gettempdir()) / ("latitude-demo-" + plan["uuid"]) / str(rank)
    local.mkdir(parents=True, exist_ok=True)
    downloads = plan["input_downloads"]
    if downloads["manifest.json"]["sha256"] != plan["input_manifest_sha256"]:
        raise RuntimeError("冻结下载清单身份不匹配")
    for name, item in downloads.items():
        if Path(name).name != name or urlparse(item["url"]).scheme != "https":
            raise RuntimeError("冻结下载文件名或协议无效")
        destination = local / name
        if destination.is_file() and _demo_file_digest(destination) == item["sha256"]:
            continue
        staging = destination.with_name(destination.name + ".staging")
        if staging.exists():
            raise RuntimeError("节点存在未完成输入下载；保留现场并结束")
        digest = hashlib.sha256()
        with urlopen(item["url"], timeout=60) as response, staging.open("xb") as stream:
            for block in iter(lambda: response.read(8 * 1024 * 1024), b""):
                stream.write(block)
                digest.update(block)
            stream.flush()
            os.fsync(stream.fileno())
        if digest.hexdigest() != item["sha256"]:
            raise RuntimeError("节点本地下载内容不匹配")
        os.replace(staging, destination)
    manifest = json.loads((local / "manifest.json").read_text(encoding="utf8"))
    if manifest["identity"] != plan["identity"] or set(downloads) != {*manifest["files"], "manifest.json"}:
        raise RuntimeError("冻结下载文件集合或输入身份不匹配")
    for name, expected in manifest["files"].items():
        if expected != downloads[name]["sha256"] or _demo_file_digest(local / name) != expected:
            raise RuntimeError("节点本地输入清单与实际内容不匹配")
    return local


def _demo_node_inputs(code_root, input_dir, *, frame_names=None):
    """确认冻结身份，加载本节点所需的同一份完整输入。"""
    code_root, input_dir = Path(code_root), Path(input_dir)
    plan = json.loads((code_root / "demo_plan.json").read_text(encoding="utf8"))
    manifest_path = input_dir / "manifest.json"
    if _demo_file_digest(manifest_path) != plan["input_manifest_sha256"]:
        raise RuntimeError("输入清单身份不匹配")
    manifest = json.loads(manifest_path.read_text(encoding="utf8"))
    identity_key = hashlib.sha256(json.dumps(plan["identity"], sort_keys=True).encode()).hexdigest()
    if (identity_key != plan["identity_key"] or plan["stage"] != plan["identity"]["stage"]
            or plan["metadata"]["scope"] != plan["identity"]["scope"]
            or plan["metadata"]["upstream"] != plan["identity"]["upstream"]):
        raise RuntimeError("节点计划与冻结身份不匹配")
    if manifest["identity"] != plan["identity"]:
        raise RuntimeError("节点输入身份不匹配")
    for relative, expected in plan["identity"]["source_files"].items():
        if _demo_file_digest(code_root / relative) != expected:
            raise RuntimeError("节点代码身份不匹配")
    if sys.version_info[:2] != (3, 13):
        raise RuntimeError("云端 Python 主／次版本不符合本次 3.13 约定")
    for specification in plan["identity"]["requirements"]:
        package, expected = specification.split("==", 1)
        if importlib.metadata.version(package) != expected:
            raise RuntimeError(f"云端依赖版本不匹配：{package}")
    names = list(plan["identity"]["frames"]) if frame_names is None else list(frame_names)
    frames = {}
    for name in names:
        path = input_dir / (name + ".parquet")
        if _demo_file_digest(path) != manifest["files"][path.name]:
            raise RuntimeError("节点输入文件内容不匹配")
        encoded = (pq.ParquetFile(path).metadata.metadata or {}).get(b"ARROW:schema")
        if encoded is None:
            raise RuntimeError("冻结输入缺少原始 Arrow Schema")
        original_schema = pa.ipc.read_schema(pa.BufferReader(base64.b64decode(encoded))).remove_metadata()
        expected = plan["identity"]["frames"][name]
        if hashlib.sha256(original_schema.serialize().to_pybytes()).hexdigest() != expected["schema_sha256"]:
            raise RuntimeError("节点输入原始 Schema 不匹配")
        # Parquet 会把 timestamp[s] 读成毫秒；按保存的原 Schema 安全还原再核对内容。
        table = pq.read_table(path).cast(original_schema, safe=True).replace_schema_metadata(None).combine_chunks()
        if table.num_rows != expected["rows"]:
            raise RuntimeError("节点输入行数不匹配")
        if _demo_input_digest(table) != expected["content_sha256"]:
            raise RuntimeError("节点输入内容身份不匹配")
        frames[name] = table.to_pandas()
    return plan, frames


def _demo_save_node(code_root, output_dir, plan, rank, products):
    """写节点成果及完成记录；products 只承载已有函数返回值。"""
    node_root = Path(output_dir) / f"node_{rank}" / "project"
    for relative in plan["identity"]["source_files"]:
        target = node_root / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(Path(code_root) / relative, target)
    files = {}
    for name, item in products.items():
        frame, schema, producer, upstream, parameters, sort_keys = item
        frame = frame.copy()
        for field in schema:
            if pa.types.is_map(field.type):
                frame[field.name] = frame[field.name].astype(object).where(frame[field.name].notna(), None)
        path = node_root / plan["products"][name]
        identity = _real_demo_identity(node_root, node_root / "R04_Research/a01_Methods/b04_ReturnVolatilityModeling" / (producer + ".ipynb"),
            upstream, parameters, plan["metadata"]["scope"])
        identity["dlc_identity_key"], identity["dlc_uuid"] = plan["identity_key"], plan["uuid"]
        installed = _save_real_demo(frame, path, schema, identity, sort_keys)
        files[name] = dict(file_sha256=_demo_file_digest(path), rows=len(frame),
                           content_sha256=installed["output_content_sha256"])
    _demo_json(Path(output_dir) / f"node_{rank}" / "result.json",
               dict(state="completed", rank=rank, node_count=plan["identity"]["resources"]["instance_count"],
                    stage=plan["stage"], identity_key=plan["identity_key"], uuid=plan["uuid"], files=files))


def _demo_recover(ctx):
    """按节点身份和完整覆盖接收，保存 demo；回收失败不重算。"""
    if ctx["cached"]:
        return
    record = json.loads(ctx["record_path"].read_text(encoding="utf8"))
    if record["state"] not in ("cloud_completed_pending_recovery", "recovery_failed", "completed"):
        raise RuntimeError("云作业未成功，不接收为完整成果")
    session = ctx.get("session") or _demo_pai_session(ctx["resources"])
    uri, owned = urlparse(record["actual_output_uri"]), urlparse(record["requested_output_uri"])
    if uri.scheme != "oss" or uri.netloc != owned.netloc or not uri.path.startswith(owned.path):
        raise RuntimeError("输出不属于本作业")
    prefix = uri.path.strip("/") + "/"
    run, groups = ctx["record_path"].parent, {}
    download = run / "download"
    reference = (ctx["full_inputs"]["frames"].get("positions") if ctx["stage"] == "observations"
                 else ctx["full_inputs"]["frames"].get("observations"))
    reference_ids = None if reference is None else reference["sample_id"].tolist()
    nodes = ctx["resources"]["instance_count"]
    try:
        for rank in range(nodes):
            local_node = download / f"node_{rank}"
            local_node.mkdir(parents=True, exist_ok=True)
            result_path = local_node / "result.json"
            session.oss_bucket.get_object_to_file(prefix + f"node_{rank}/result.json", str(result_path))
            result = json.loads(result_path.read_text(encoding="utf8"))
            if (result["state"] != "completed" or result["rank"] != rank or result["node_count"] != nodes
                    or result["stage"] != ctx["stage"] or result["identity_key"] != record["identity_key"]
                    or result["uuid"] != record["uuid"]):
                raise RuntimeError("节点完成记录身份不匹配")
            expected_names = list(record["products"])
            # 季节阶段每节点输出四条曲线的独立日期，rank 0 另存完整贡献。
            if ctx["stage"] == "seasonality":
                paths = list(record["products"])
                expected_names = paths if rank == 0 else [name for name in paths if "contributions_demo" not in name]
            if set(result["files"]) != set(expected_names):
                raise RuntimeError("节点产物集合不匹配")
            node_project = local_node / "project"
            for relative, content in ctx["source_files"].items():
                target = node_project / relative
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(content)
            for name, item in result["files"].items():
                path = node_project / record["products"][name]
                path.parent.mkdir(parents=True, exist_ok=True)
                session.oss_bucket.get_object_to_file(prefix + f"node_{rank}/project/" + record["products"][name], str(path))
                if _demo_file_digest(path) != item["file_sha256"]:
                    raise RuntimeError("下载文件摘要不匹配")
                frame, identity = _read_real_demo(path)
                if (len(frame) != item["rows"] or identity["output_content_sha256"] != item["content_sha256"]
                        or identity.get("dlc_identity_key") != record["identity_key"] or identity.get("dlc_uuid") != record["uuid"]):
                    raise RuntimeError("节点成果内容／身份不匹配")
                if reference_ids is not None:
                    expected_ids = (np.array_split(np.asarray(reference_ids), nodes)[rank].tolist()
                                    if ctx["stage"] == "observations" else reference_ids[rank::nodes])
                    if frame["sample_id"].tolist() != expected_ids:
                        raise RuntimeError("节点目标覆盖错误")
                if ctx["stage"] == "seasonality":
                    if "contributions_demo" in name:
                        main = ctx["full_inputs"]["frames"]["main"]
                        if len(frame) != len(main) or frame["bar_at"].tolist() != main["bar_at"].tolist():
                            raise RuntimeError("分钟贡献覆盖不完整")
                    else:
                        calendar = ctx["full_inputs"]["frames"]["calendar"]
                        dates = pd.DatetimeIndex(pd.to_datetime(calendar["trading_date"]).unique()).sort_values()
                        expected_dates = [pd.Timestamp(day).date().isoformat() for day in np.array_split(dates, nodes)[rank]]
                        actual_dates = sorted(pd.to_datetime(frame["trading_date"]).dt.date.astype(str).unique().tolist())
                        if actual_dates != expected_dates:
                            raise RuntimeError("季节曲线节点日期覆盖错误")
                groups.setdefault(name, []).append((frame, identity, path))
        if set(groups) != set(record["products"]):
            raise RuntimeError("完整产物集合缺失")
        installed = {}
        accepted_root = run / "accepted/project"
        for relative, content in ctx["source_files"].items():
            target = accepted_root / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
        for name, parts in groups.items():
            if len(parts) != (1 if ctx["stage"] == "seasonality" and "contributions_demo" in name else nodes):
                raise RuntimeError("产物节点数量不匹配")
            frame = pd.concat([p[0] for p in parts], ignore_index=True)
            identity = {k: v for k, v in parts[0][1].items() if not k.startswith("output_") and k != "generated_at_utc"}
            if reference_ids is not None:
                frame = frame.sort_values("sample_id").reset_index(drop=True)
                if frame["sample_id"].tolist() != reference_ids:
                    raise RuntimeError("合并结果缺失、重复或错位")
            identity["parameters"] = {k: v for k, v in identity["parameters"].items() if k != "target_partition"}
            for _, other, _ in parts:
                comparable = {k: v for k, v in other.items() if not k.startswith("output_") and k != "generated_at_utc"}
                comparable["parameters"] = {k: v for k, v in comparable["parameters"].items() if k != "target_partition"}
                if comparable != identity:
                    raise RuntimeError("节点计算条件不同")
            if ctx["stage"] == "seasonality" and "contributions_demo" not in name:
                if frame.duplicated(["trading_date", "minute_slot"]).any():
                    raise RuntimeError("季节曲线存在重复位置")
            encoded = pq.ParquetFile(parts[0][2]).metadata.metadata[b"ARROW:schema"]
            schema = pa.ipc.read_schema(pa.BufferReader(base64.b64decode(encoded))).remove_metadata()
            sort_keys = parts[0][1]["output_sort_keys"]
            staged = accepted_root / record["products"][name]
            _save_real_demo(frame, staged, schema, identity, sort_keys)
            _, checked = _read_real_demo(staged)
            destination = ctx["root"] / record["products"][name]
            destination.parent.mkdir(parents=True, exist_ok=True)
            if destination.exists() and _demo_file_digest(destination) != _demo_file_digest(staged):
                # 本次完整计算的旧成果保留字节，不能将旧来源改成新来源。
                previous = run / "replaced_previous" / record["products"][name]
                previous.parent.mkdir(parents=True, exist_ok=True)
                if not previous.exists():
                    shutil.copyfile(destination, previous)
            installation = destination.with_name(destination.name + ".installing")
            if installation.exists():
                raise RuntimeError(f"存在未处理的安装文件：{installation}")
            shutil.copyfile(staged, installation)
            _read_real_demo(installation)
            os.replace(installation, destination)
            _read_real_demo(destination)
            installed[name] = dict(file_sha256=_demo_file_digest(destination), rows=checked["output_rows"],
                                   content_sha256=checked["output_content_sha256"],
                                   failed_rows=int(frame["status"].ne("ok").sum()) if "status" in frame else None)
        _demo_json(run / "completion.json", dict(state="completed", uuid=record["uuid"],
            identity=ctx["identity"], products=installed, completed_at_utc=datetime.now(timezone.utc).isoformat()))
        record["state"] = "completed"
        _demo_json(ctx["record_path"], record)
        ctx["record"], ctx["cached"] = record, True
    except BaseException as error:
        record.update(state="recovery_failed", recovery_error=f"{type(error).__name__}: {error}")
        _demo_json(ctx["record_path"], record)
        raise


def _demo_read_saved(root, stage, owner):
    """独立展示只复读已验收成果；不调用认证、提交、查询或计算。"""
    root = Path(root)
    frames = {}
    for name, relative in _demo_products(stage).items():
        if owner in name:
            frame, identity = _read_real_demo(root / relative)
            key, request_uuid = identity["dlc_identity_key"], identity["dlc_uuid"]
            record_owner = {"seasonality": "c01_IntradaySeasonality", "observations": "c02_ModelObservations",
                            "models": "c03_ReturnModels"}[stage]
            run = root / "R04_Research/a01_Methods/b04_ReturnVolatilityModeling" / (record_owner + "_CloudRuns") / key / request_uuid
            completion = json.loads((run / "completion.json").read_text(encoding="utf8"))
            record = json.loads((run / "job.json").read_text(encoding="utf8"))
            if (completion["state"] != "completed" or completion["uuid"] != request_uuid
                    or completion["identity"] != record["identity"]
                    or set(completion["products"]) != set(_demo_products(stage))):
                raise RuntimeError("完整 demo 完成记录无效")
            for product, item in completion["products"].items():
                if _demo_file_digest(root / record["products"][product]) != item["file_sha256"]:
                    raise RuntimeError("完整 demo 产物缺失或内容变化")
            frames[name] = frame
    return frames
def _seasonality_demo_chunk(code_root, input_dir, rank, node_count, measure):
    """按独立输出日期分工；窗口历史与初始回填仍使用完整输入。"""
    plan, frames = _demo_node_inputs(code_root, input_dir, frame_names=("main", "calendar"))
    parameters = plan["identity"]["parameters"]
    computed = _compute_intraday_demo(frames["main"], frames["calendar"],
        start=parameters["start"], test_start=parameters["test_start"], measures=())
    contributions = computed["contributions"]
    all_dates = pd.DatetimeIndex(pd.to_datetime(frames["calendar"]["trading_date"]).unique()).sort_values()
    output_dates = np.array_split(all_dates, node_count)[rank]
    diagnostics = {}
    if len(output_dates):
        causal = estimate_intraday_seasonality(contributions, frames["calendar"],
            contribution_col=f"{measure}_contribution",
            lookbacks={"1y": pd.DateOffset(years=1), "3y": pd.DateOffset(years=3)},
            output_dates=output_dates, huber_t=parameters["huber_t"],
            huber_tol=parameters["huber_tol"], huber_maxiter=parameters["huber_maxiter"])
        filled = backfill_initial_seasonality(causal, contributions, frames["calendar"],
            train_start=parameters["start"], test_start=parameters["test_start"])
        assert filled["diagnostics"].loc[filled["diagnostics"]["is_initial_backfill"], "history_end"].dropna().lt(
            pd.Timestamp(parameters["test_start"])).all()
        for window, curve in filled["curves"].items():
            pd.testing.assert_frame_equal(curve.loc[parameters["test_start"]:],
                                          causal["curves"][window].loc[parameters["test_start"]:])
            np.testing.assert_allclose(curve.mean(axis=1).dropna(), 1.0, rtol=1e-10, atol=1e-10)
            rows = filled["diagnostics"].loc[filled["diagnostics"]["window_name"].eq(window)].copy()
            positions = curve.index.get_indexer(pd.DatetimeIndex(rows["trading_date"]))
            slots = curve.columns.get_indexer(rows["minute_slot"])
            rows["seasonal_factor"] = curve.to_numpy()[positions, slots]
            rows["causal_factor"] = causal["curves"][window].to_numpy()[positions, slots]
            diagnostics[window] = rows
    else:
        diagnostics = {w: pd.DataFrame(columns=computed["curve_schema"].names) for w in ("1y", "3y")}
    return dict(diagnostics=diagnostics, curve_schema=computed["curve_schema"],
                contributions=contributions if rank == 0 and measure == "rv" else None,
                contribution_schema=computed["contribution_schema"],
                output_dates=[pd.Timestamp(day).date().isoformat() for day in output_dates])


def run_seasonality_demo_node(code_root, input_dir, output_dir, *, rank, node_count):
    """十节点分日期，每节点两个 CPU 独立估计 RV／BPV。"""
    input_dir = _demo_stage_node_inputs(code_root, input_dir, rank)
    plan, _ = _demo_node_inputs(code_root, input_dir, frame_names=())
    if node_count != plan["identity"]["resources"]["instance_count"] or node_count != 10 or not 0 <= rank < node_count:
        raise ValueError("节点身份与完整配置不匹配")
    with ProcessPoolExecutor(max_workers=2, mp_context=multiprocessing.get_context("spawn")) as pool:
        futures = [pool.submit(_seasonality_demo_chunk, code_root, input_dir, rank, node_count, measure)
                   for measure in ("rv", "bpv")]
        parts = [future.result() for future in futures]
    products, names = {}, list(_demo_products("seasonality"))
    parameters, upstream = plan["identity"]["parameters"], plan["metadata"]["upstream"]
    if rank == 0:
        products[names[0]] = (parts[0]["contributions"], parts[0]["contribution_schema"], "c01_IntradaySeasonality",
            upstream, dict(price="adjusted_log_close", adjacency="ordinary_rows",
                selection_trading_dates=plan["metadata"]["scope"]["minute_read_trading_dates"]),
            [("bar_at", "ascending")])
    for index, (measure, window) in enumerate([(m, w) for m in ("rv", "bpv") for w in ("1y", "3y")], 1):
        part = parts[0 if measure == "rv" else 1]
        partition = dict(rank=rank, node_count=node_count, selection="trading_date_contiguous",
                         output_dates=part["output_dates"])
        products[names[index]] = (part["diagnostics"][window], part["curve_schema"], "c01_IntradaySeasonality",
            upstream, dict(contribution=measure, lookback_years=int(window[:-1]),
                huber_t=parameters["huber_t"], huber_tol=parameters["huber_tol"], huber_maxiter=parameters["huber_maxiter"],
                train_start=parameters["start"], test_start=parameters["test_start"], target_partition=partition),
            [("trading_date", "ascending"), ("minute_slot", "ascending")])
    _demo_save_node(code_root, output_dir, plan, rank, products)

def _demo_load_positions(root):
    """只读既有完整位置，在其记录的线程环境下严格核对身份；不重新校准。"""
    from R04_Research.a01_Methods.b03_Sampling.c02_SamplingExecution import read_reusable_sampling_identity
    import numba
    root = Path(root)
    sampling_base = root / "R04_Research/a01_Methods/b03_Sampling"
    path = sampling_base / "c01_SamplingPositions_DemoData/c01_SamplingPositions_RB_train20100101-20241231_test20250101-20260930_N15_gk_money_all_contracts_log_interpolate_fixed_demo.parquet"
    adjustment = root / "R04_Research/a01_Methods/b02_MainContinuousAdjustment/c01_MainContinuousAdjustment_DemoData/c01_MainContinuousAdjustment_RB_train20100101-20241231_test20250101-20260930_ratio1.10_log_common_demo.parquet"
    saved_identity = json.loads(pq.read_schema(path).metadata[b"demo_identity"])
    previous_threads = numba.get_num_threads()
    try:
        numba.set_num_threads(saved_identity["runtime"]["threads"])
        identity = read_reusable_sampling_identity(path, project_root=root,
            producer_notebook=sampling_base / "c01_SamplingPositions.ipynb", upstream_path=adjustment,
            underlying_code="RB", test_start_date="2025-01-01",
            parameters=dict(N=15, axis="gk_money", quantity_scope="all_contracts",
                sampling_method="log_interpolate", execution_mode="fixed", K=None))
    finally:
        numba.set_num_threads(previous_threads)
    if identity is None:
        raise RuntimeError("既定 N15 采样位置身份不匹配；请先完成 b03 对应 demo，不在此重新采样")
    return pq.read_table(path).to_pandas(), identity


def _demo_read_trial_sources(root):
    """完整上游成果未就绪时，只为内存试错读取全量真实分钟及既定位置。"""
    raw = _load_intraday_demo_inputs(root, start="2010-01-01", end="2026-09-30", test_start="2025-01-01")
    positions, identity = _demo_load_positions(root)
    return dict(main=raw["main"], calendar=raw["calendar"], positions=positions,
                scope=raw["scope"], upstream=dict(**raw["upstream"], positions=identity))


def _demo_trial_prefix(sources, *, include_seasonality, points=64, days=8):
    """只切真实前缀并调用已有函数；不保存、不把试错数据当作完整成果。"""
    positions = sources["positions"].iloc[:points].copy()
    if len(positions) < 2:
        raise ValueError("试错至少需要两个既定采样位置")
    required_stop = int(positions["source_minute_index"].max()) + 1
    dates = sorted(pd.to_datetime(sources["main"]["trading_date"]).unique())
    requested_last = pd.Timestamp(sources["main"].iloc[required_stop - 1]["trading_date"])
    selected_dates = [day for i, day in enumerate(dates) if i < days or pd.Timestamp(day) <= requested_last]
    if len(selected_dates) < 3:
        raise ValueError("至少需要三个真实交易日进行因果及回填试错")
    main = sources["main"].loc[pd.to_datetime(sources["main"]["trading_date"]).isin(selected_dates)].copy()
    calendar = sources["calendar"].loc[pd.to_datetime(sources["calendar"]["trading_date"]).isin(selected_dates)].copy()
    computed = _compute_intraday_demo(main, calendar, start=pd.Timestamp(selected_dates[0]),
        test_start=pd.Timestamp(selected_dates[-1]), measures=("rv", "bpv") if include_seasonality else ())
    if required_stop > len(computed["contributions"]):
        raise RuntimeError("试错分钟前缀未覆盖所选采样位置")
    return dict(positions=positions, computed=computed)


In [ ]:
_demo_upload(demo_context)  # 只冻结同一份 full_inputs，不重新读取上游；不上传试错结果。


In [ ]:
demo_job = _demo_submit(demo_context)
print(f'{demo_job["name"]} | {demo_job.get("job_id", "本地成果复用")} | {demo_job.get("console_uri", "")}')
print("输出位置：", demo_job.get("actual_output_uri", demo_job.get("requested_output_uri", "")))


In [ ]:
_demo_wait(demo_context)  # 普通 Python 每 60 秒查同一 ID；明确失败后留证并结束，不重提。


In [ ]:
_demo_recover(demo_context)  # 接收所有节点、核对身份／完整覆盖、安装并复读；失败不重新计算。
print("完整 demo 数据已保存；展示可独立执行最后一格。")


In [ ]:
from pathlib import Path
import sys
from IPython.display import display
candidate_root = Path.cwd().resolve()
while not (candidate_root / "config/settings.py").is_file():
    if candidate_root == candidate_root.parent:
        raise RuntimeError("找不到项目根目录；请在本项目内打开 Notebook")
    candidate_root = candidate_root.parent
sys.path[:0] = [str(candidate_root)]
from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import _demo_read_saved
demo_display_labels = {
    "sample_id": "sample_id（采样编号）", "sample_at": "sample_at（采样时刻）",
    "trading_date": "trading_date（交易日）", "sample_side": "sample_side（训练／测试侧）",
    "model": "model（模型）", "status": "status（状态）", "reason": "reason（原因）",
    "mean_forecast": "mean_forecast（下一步均值预测）",
    "variance_forecast": "variance_forecast（下一步条件方差）",
    "training_points": "training_points（训练采样点数）",
    "update_status": "update_status（观测更新状态）", "updated_mean_forecast": "updated_mean_forecast（更新后的下一步预测）",
    "parameters": "parameters（具名拟合系数）", "features": "features（HAR 特征）",
    "components": "components（MEM 成分）", "log_return": "log_return（区间对数收益）",
    "rv": "rv（区间平方收益贡献）", "bpv": "bpv（区间双幂变差贡献）",
    "log_return_status": "log_return_status（收益状态）", "rv_status": "rv_status（RV 状态）",
    "bpv_status": "bpv_status（BPV 状态）", "bar_at": "bar_at（分钟末时刻）",
    "rv_contribution": "rv_contribution（分钟平方收益贡献）",
    "bpv_contribution": "bpv_contribution（分钟双幂变差贡献）", "minute_slot": "minute_slot（分钟位置）",
    "seasonal_factor": "seasonal_factor（季节因子）", "causal_factor": "causal_factor（因果季节因子）",
    "is_initial_backfill": "is_initial_backfill（是否初始回填）"}
saved_demo_tables = _demo_read_saved(candidate_root, "seasonality", "c01_IntradaySeasonality")
for filename, saved_frame in saved_demo_tables.items():
    print(filename, f"；完整 {len(saved_frame):,} 行")
    selected_columns = [name for name in demo_display_labels if name in saved_frame.columns]
    display(saved_frame.loc[:, selected_columns].head(8).rename(columns=demo_display_labels))
    if "status" in saved_frame:
        display(saved_frame.groupby("status", dropna=False).size().rename("行数").rename_axis("status（状态）").to_frame())
# 本主题当前提供表格展示；若增加正式绘制函数，在此调用即可，不触发计算或保存试错图。
